In [ ]:
import numpy as np 
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

**Feature Engineering**

In [8]:
df = pd.read_csv("aapl.us.txt" , parse_dates=['Date'] , index_col='Date')

#creat target (Y): tommorow's price
df['Target_price_change'] = df['Close'].shift(-1) - df['Close']

#lag features
df['Close_lag1'] = df['Close'].shift(1) #yesterday close
df['Close_lag2'] = df['Close'].shift(2) #2 days ago close
df['Volume_lag1'] = df['Volume'].shift(3) #yesterday lag

#Calculate RSI (Relative Strength Index) Feature
delta = df['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI_14'] = 100 - (100 / (1 + rs))

# 5. Calculate MACD (Moving Average Convergence Divergence) Feature
exp1 = df['Close'].ewm(span=12, adjust=False).mean()
exp2 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = exp1 - exp2

#drop nan values
df_ml = df.dropna().copy()

print(f"Dataset shape after feature engineering: {df_ml.shape}")
df_ml[['Close', 'Target_price_change', 'Close_lag1', 'RSI_14', 'MACD']].head()

Dataset shape after feature engineering: (8350, 12)


,Close,Target_price_change,Close_lag1,RSI_14,MACD
Date,,,,,
1984-09-26,0.41111,0.00000,0.41618,44.335019,-0.000606
1984-09-27,0.41111,-0.01030,0.41111,44.335019,-0.001792
1984-09-28,0.40081,-0.00895,0.41111,41.479206,-0.003523
1984-10-01,0.39186,0.00257,0.40081,34.737966,-0.005552
1984-10-02,0.39443,0.00638,0.39186,40.244012,-0.006874


In [10]:
f_cols = ['Close', 'Target_price_change', 'Close_lag1', 'RSI_14', 'MACD']
x = df_ml[f_cols]
y = df['Target_price_change']

#split based on cut off date
cutoff_date = '2015-01-01'
x_train = x.loc[:cutoff_date]
y_train = y.loc[:cutoff_date]

x_test = x.loc[cutoff_date:]
y_test = y.loc[cutoff_date:]

print(f"Training set size: {x_train.shape[0]} rows (from {x_train.index.min().date()} to {x_train.index.max().date()})")
print(f"Testing set size: {x_test.shape[0]} rows (from {x_test.index.min().date()} to {x_test.index.max().date()})")

Training set size: 7629 rows (from 1984-09-26 to 2014-12-31)
Testing set size: 721 rows (from 2015-01-02 to 2017-11-09)
